# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset in Croissant format using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")


## 2. Data Overview

Review available record sets and fields. All entities are referenced by their `@id`.

Let's inspect the metadata object to list available record sets, their `@id`, and the fields and columns defined.

In [ ]:
# Show dataset metadata structure for exploration
from pprint import pprint

record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")

# Print each record set's @id, name, available fields, and columns
for rs in record_sets:
    print(f"Record set @id: {rs.id_}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.id_}: {field.name}")
    if hasattr(rs, 'columns') and getattr(rs, 'columns', None):
        print(f"  Columns (for tabular data):")
        for col in rs.columns:
            print(f"    - {col.id_}: {col.name}")
    print()


### Preview a few records from a record set

You can preview actual records from a chosen record set by using its `@id`. Here we will automatically attempt to show the first record set for demonstration.

In [ ]:
# Pick the first record set @id (update if working with a specific one)
if record_sets:
    first_record_set_id = record_sets[0].id_
    print(f"Previewing records from record set: {first_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record sets found!")

## 3. Data Extraction

Load data from specific record sets into pandas DataFrames for analysis. Here, we extract all available record sets from the dataset, referencing them by their `@id`.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id_ for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}.")
    else:
        print(f"No records found for {record_set_id}.")

if dataframes:
    # Pick the first dataframe for demonstration
    demo_rs_id = list(dataframes.keys())[0]
    print(f"Columns in {demo_rs_id}:\n{dataframes[demo_rs_id].columns.tolist()}")
    display(dataframes[demo_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply analytical steps, such as filtering records, normalizing numeric fields, and categorizing data. Here, we will attempt numerical field EDA on the first tabular record set.

In [ ]:
# Automatic numeric field example for demo
import numpy as np

if dataframes:
    demo_df = dataframes[demo_rs_id]
    # Try to find a numeric column
    numeric_cols = demo_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # You may change to an appropriate field
        print(f"Numeric field selected: {numeric_field}")
        threshold = demo_df[numeric_field].mean()  # Example threshold: mean
        filtered_df = demo_df[demo_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a string/categorical column
        group_fields = demo_df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field, dropna=True)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields found in the dataframe.")
else:
    print("No dataframes to analyze.")

## 5. Visualization

Visualize distributions or relationships in the data using matplotlib or seaborn. We plot a histogram and a scatterplot if numeric data allows.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    # Histogram of the main numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(demo_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # If there's a second numeric field, plot scatter
    if len(numeric_cols) > 1:
        plt.figure(figsize=(7,5))
        sns.scatterplot(
            x=demo_df[numeric_cols[0]],
            y=demo_df[numeric_cols[1]]
        )
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion

This notebook provided a guided exploration of a FAIR^2 Croissant dataset using the `mlcroissant` library. We reviewed its structure via `@id` fields, extracted records, performed basic analysis, and visualized distributions. For more advanced analysis or domain-specific questions, tailor the analyses to your research needs!